Step 1: Define File Path and Load Data
In this step, we define the file path for the 2017 property assessment dataset and check if the file exists. If found, we load the dataset, remove any leading/trailing whitespace from column names, and handle potential missing values.

In [2]:
import os
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

In [3]:
# Define the file path
file_path = r'C:\Users\sul19\Desktop\701 Project\Property Assessment Datasets\Property Assessment 2017 V1.xlsx'

# Check if the file exists before attempting to load it
if os.path.exists(file_path):
    print("File found! Proceeding to load the data.")
    
    # Attempt to load the dataset using 'openpyxl' engine
    df = pd.read_excel(file_path, engine='openpyxl', na_values=['', ' '])

    # Strip any leading/trailing whitespace from the column names (just in case)
    df.columns = df.columns.str.strip()
    
else:
    print(f"File not found at {file_path}. Please check the file path.")

File found! Proceeding to load the data.


Step 2: Clean ZIP_CODE Column and Standardize OVERALL_COND Column
This step standardizes the ZIP_CODE column by removing leading zeros and trailing underscores to ensure consistency. It then replaces shorthand codes in the OVERALL_COND column with descriptive labels, removes invisible characters, and handles blank or missing values to improve data quality.

In [9]:
# Remove leading zeros and trailing underscores from the 'ZIP_CODE' column
df['ZIP_CODE'] = df['ZIP_CODE'].astype(str).str.lstrip('0').str.rstrip('_')

# Replace specific values in the 'OVERALL_COND' column before grouping the data
df['OVERALL_COND'] = df['OVERALL_COND'].replace({
    'A': 'A - Average',
    'G': 'G - Good',
    'E': 'E - Excellent',
    'F': 'F - Fair',
    'P': 'P - Poor'
})

# Replace non-breaking spaces and other invisible characters in the overall condition column
df['OVERALL_COND'] = df['OVERALL_COND'].astype(str).str.replace('\u00A0', '').str.strip()

# Replace any blank or missing values with 'none'
df['OVERALL_COND'] = df['OVERALL_COND'].replace(r'^\s*$', 'none', regex=True)

# Replace specific values in OVERALL_COND
df['OVERALL_COND'] = df['OVERALL_COND'].replace({
    'AVG - Default - Average': 'A - Average',
    'EX - Excellent': 'E - Excellent'
}, regex=False)

Step 3: Group Data by ZIP_CODE and Summarize OVERALL_COND Counts
We group data by ZIP code and count occurrences of each condition to create a summary that highlights the distribution of housing conditions across ZIP codes.

In [10]:
# Group data by 'ZIP_CODE' and count the occurrences of each condition in the column
overall_cond_summary = df.groupby('ZIP_CODE')['OVERALL_COND'].value_counts().unstack().fillna(0)

# Display the result of the analysis
print("Housing condition summary by ZIP code:")
#print(condition_summary)
print(overall_cond_summary)

Housing condition summary by ZIP code:
OVERALL_COND  A - Average  E - Excellent  F - Fair  G - Good  P - Poor  \
ZIP_CODE                                                                 
                      0.0            0.0       0.0       0.0       0.0   
2090                  0.0            0.0       0.0       0.0       0.0   
2108                 40.0           87.0       3.0     128.0       1.0   
2109                 20.0            1.0       1.0       2.0       0.0   
2110                  0.0            0.0       0.0       0.0       0.0   
2111                 12.0            0.0       8.0       2.0       0.0   
2112                  0.0            0.0       0.0       0.0       0.0   
2113                 90.0            2.0      11.0      30.0       1.0   
2114                 89.0           26.0      10.0     129.0       2.0   
2115                 85.0            8.0       2.0      77.0       0.0   
2116                172.0          104.0      18.0     194.0       7.0   

Step 4: Map Condition Labels to Numeric Values for Analysis
To analyze conditions quantitatively, this step maps condition labels to numeric values. We then calculate the mean condition score for each ZIP code to assess average housing conditions.

Step 5: Convert Mean Condition Scores to Descriptive Labels
This step converts mean condition scores back to descriptive labels, making the results easier to interpret. Each ZIP code’s housing condition is assigned a descriptive label.

In [11]:
# Function to assign numerical values to conditions 
def condition_to_numeric(cond):
    mapping = {
        'E - Excellent': 5,
        'VG - Very Good': 4,
        'G - Good': 3.5,
        'A - Average': 3,
        'F - Fair': 2,
        'P - Poor': 1.5,
        'VP - Very Poor': 1,
        'US - Unsound': 0,
        
        
        'none': np.nan  # Treat 'none' as NaN for numerical purposes
    }
    return mapping.get(cond, np.nan)

# Apply the mapping to calculate average conditions
df['OVERALL_COND_NUM'] = df['OVERALL_COND'].apply(condition_to_numeric)

# Group by ZIP_CODE and calculate the mean, count, and standard deviation
condition_analysis = df.groupby('ZIP_CODE').agg(
    overall_cond_mean=('OVERALL_COND_NUM', 'mean'),
).reset_index()

# Display the summary
print("Housing condition analysis by ZIP code:")
print(condition_analysis)

def mean_to_condition(mean_score):
    if mean_score >= 4.75:
        return 'Excellent'
    elif mean_score >= 4:
        return 'Very Good'
    elif mean_score >= 3.5:
        return 'Good'
    elif mean_score >= 3:
        return 'Average'
    elif mean_score >= 2:
        return 'Fair'
    elif mean_score >= 1:
        return 'Poor'
    elif mean_score >= 0.5:
        return 'Very Poor'
    
    else:
        return 'Unsound'

# Apply the function to map the means back to descriptive condition labels
condition_analysis['overall_cond_label'] = condition_analysis['overall_cond_mean'].apply(mean_to_condition)

# Print the result for each ZIP code
print("Overall Condition Analysis by ZIP Code:")
print(condition_analysis[['ZIP_CODE', 'overall_cond_mean', 'overall_cond_label']])

Housing condition analysis by ZIP code:
   ZIP_CODE  overall_cond_mean
0                          NaN
1      2090                NaN
2      2108           3.901544
3      2109           3.083333
4      2110                NaN
5      2111           2.681818
6      2112                NaN
7      2113           3.048507
8      2114           3.404297
9      2115           3.305233
10     2116           3.558586
11     2118           3.400685
12     2119           3.074674
13     2120           3.097685
14     2121           3.055323
15     2122           3.037222
16     2124           3.058581
17     2125           3.057621
18     2126           3.034543
19     2127           3.095443
20     2128           3.047196
21     2129           3.169212
22     2130           3.134422
23     2131           3.047623
24     2132           3.068708
25     2133                NaN
26     2134           3.023487
27     2135           3.034757
28     2136           3.037062
29     2137                NaN

Step 6: Standardize YR_BUILT and YR_REMODEL Columns
This step ensures the YR_BUILT and YR_REMODEL columns are numeric. Any values beyond 2024 are replaced with NaN to avoid future-dated entries, and we calculate the mean construction and remodel years by ZIP code.

Step 7: Classify Buildings by Age
In this step, we classify buildings based on their construction or remodel year into categories such as 'Old,' 'Average,' or 'New,' providing insight into the age distribution within each ZIP code.

In [12]:
# Ensure YR_BUILT and YR_REMODEL are numeric and replace years > 2024 with NaN
df['YR_BUILT'] = pd.to_numeric(df['YR_BUILT'], errors='coerce')
df['YR_REMODEL'] = pd.to_numeric(df['YR_REMODEL'], errors='coerce')
df['YR_BUILT'] = df['YR_BUILT'].apply(lambda x: x if x <= 2024 else np.nan)
df['YR_REMODEL'] = df['YR_REMODEL'].apply(lambda x: x if x <= 2024 else np.nan)

# Group by ZIP_CODE and calculate the mean for YR_BUILT and YR_REMODEL
condition_analysis = df.groupby('ZIP_CODE').agg(
    yr_built_mean=('YR_BUILT', 'mean'),
    yr_remodel_mean=('YR_REMODEL', 'mean')
).reset_index()

# Define thresholds for building classification
def classify_building(yr_built, yr_remodel, old_threshold=1970, new_threshold=2000):
    """
    Classify building as 'Old', 'Average', or 'New' based on YR_REMODEL or YR_BUILT.
    If YR_REMODEL exists, use it; otherwise, use YR_BUILT.
    """
    if not pd.isna(yr_remodel):
        year = yr_remodel  # Use YR_REMODEL if available
    else:
        year = yr_built  # Otherwise, use YR_BUILT
    
    if pd.isna(year):
        return 'Unknown'
    elif year <= old_threshold:
        return 'Old'
    elif year >= new_threshold:
        return 'New'
    else:
        return 'Average'
    
# Apply classification based on YR_BUILT and YR_REMODEL
condition_analysis['building_classification'] = condition_analysis.apply(
    lambda row: classify_building(row['yr_built_mean'], row['yr_remodel_mean']), axis=1)

# Print the classification results based on the YR_BUILT and YR_REMODEL means
print("Building Classification Based on YR_BUILT and YR_REMODEL:")
print(condition_analysis[['ZIP_CODE', 'yr_built_mean', 'yr_remodel_mean', 'building_classification']])

Building Classification Based on YR_BUILT and YR_REMODEL:
   ZIP_CODE  yr_built_mean  yr_remodel_mean building_classification
0               379.800000              NaN                     Old
1      2090    1988.000000      1988.000000                 Average
2      2108    1546.062169      1903.476443                     Old
3      2109    1748.591058      1589.127517                     Old
4      2110    1504.991736      1033.108084                     Old
5      2111    1709.665331      1847.158131                     Old
6      2112       0.000000              NaN                     Old
7      2113    1813.073193      1899.400308                     Old
8      2114    1658.672109      1832.363509                     Old
9      2115    1654.621196      1867.247363                     Old
10     2116    1818.239503      1853.205006                     Old
11     2118    1791.246679      1893.950427                     Old
12     2119    1446.915747      1249.618192               

Step 8: Include Address Details and Finalize Output
This final step incorporates street address details into the main DataFrame to create a comprehensive summary. We merge overall condition labels, assign a constant YEAR value, and save the output as an Excel file for further use.

In [13]:
# Assuming 'ST_NUM' and 'ST_NAME' are part of the original dataset
# Extract those columns from the original dataset
st_num_name = df[['ZIP_CODE', 'ST_NUM', 'ST_NAME']].drop_duplicates()

# Merge 'st_num_name' with 'condition_analysis' to include 'ST_NUM' and 'ST_NAME' with the results
condition_analysis = pd.merge(condition_analysis, st_num_name, on='ZIP_CODE', how='left')

# Add overall condition label based on previous analysis
# Ensure that the column 'overall_cond_label' from earlier condition analysis is merged correctly
overall_cond_summary = df.groupby('ZIP_CODE').agg(
    overall_cond_mean=('OVERALL_COND_NUM', 'mean'),
).reset_index()

# Reapply the condition label mapping function to map numeric values to condition labels
def mean_to_condition(mean_score):
    if mean_score >= 4.75:
        return 'Excellent'
    elif mean_score >= 4:
        return 'Very Good'
    elif mean_score >= 3.5:
        return 'Good'
    elif mean_score >= 3:
        return 'Average'
    elif mean_score >= 2:
        return 'Fair'
    elif mean_score >= 1:
        return 'Poor'
    elif mean_score >= 0.5:
        return 'Very Poor'
    else:
        return 'Unsound'

overall_cond_summary['overall_cond_label'] = overall_cond_summary['overall_cond_mean'].apply(mean_to_condition)

# Merge overall condition labels with the condition_analysis DataFrame
condition_analysis = pd.merge(condition_analysis, overall_cond_summary[['ZIP_CODE', 'overall_cond_label']], on='ZIP_CODE', how='left')

# Assign a constant value for 'YEAR'
condition_analysis['YEAR'] = 2017

# Rearranging the columns as requested
final_output = condition_analysis[['YEAR', 'ST_NUM', 'ST_NAME', 'ZIP_CODE', 'overall_cond_label', 'building_classification']]

# Save the final result to a new Excel file
output_file_path = 'Property_Assessment_2017_Output.xlsx'
final_output.to_excel(output_file_path, index=False)

print(f"File saved successfully to {output_file_path}")

File saved successfully to Property_Assessment_2017_Output.xlsx
